In [45]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict , Annotated
from pydantic import BaseModel, Field
import os
import operator

load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)


In [46]:
class EvaluationSchema(BaseModel):
    score: float = Field(description="Score out of 10", ge=0, le=10)
    feedback: str = Field(description="Detailed feedback for the essay")


In [47]:
structured_model = model.with_structured_output(EvaluationSchema)

In [48]:
essay = """ While bringing out the orders for my party of six, I set down a cup of soup in front of one of the women. Virginia glanced down at the clam chowder, and then scowled up at me.
“Hannah, I asked for a half cup of soup,” she said, sounding as if the world was ending. “You are always so slow. I don’t know how you always mess up our orders.”
I politely mumbled an apology, hoping the rest of her group did not notice how upset I was or how personally I took her comment. It was only a cup of soup, and an honest mistake, but I felt like such a disaster.
Residents at the high-end senior living facility where I waited tables several nights a week expected nothing less than a five-star dining experience and my small mistake was not to be tolerated.
When I first started working at Shannondell last summer, I was already shy and couldn’t stand the idea that someone didn’t like me. Early on, when residents would scold or criticize me, I felt like crawling under a rock. Even when I clearly was in the right, I would bite my lip and try harder to please them.
The most difficult guests by far were a group of six well-heeled women who the servers nicknamed “The Party.” They came in every night, dressed to the nines, decked with diamonds and attitude. As the staff stood by the podium waiting for residents to arrive, every server in line prayed that the hostess would not call out their name.
At first, “The Party” seemed friendly. They seemed to want to get to know me personally, compared to the other residents who would barely say anything at all. But this was all for show. They were rude and demanding. Special orders were a daily occurrence: a rare end piece of prime rib or a chicken Caesar salad without lettuce. They snapped their fingers and tapped their silverware on glasses to get my attention. I would leave the dining room distraught almost every night.
Senior residents, in general, had a difficult time making a decision, either figuring out what to order or sometimes forgetting what they had ordered once it arrived. I used to get annoyed because twenty minutes to take a dessert order seemed excessive and unnecessary. With experience, though, I’m learning patience and compassion.
It must be very difficult for residents to feel as though they were losing their independence. They used to be able to get around easily, but many of them were pushing walkers or confined to wheelchairs. I realized that many cared for others their whole lives and now it was hard to accept others caring for them.
Somehow those rough nights started to change me. Without really trying, I have become more outgoing and self-assured. The residents depend on me and my “confident” smile. Some days, I’m the one person with whom they can share stories of their past. I used to have trouble speaking in front of a group and would be shy when I did. Now, I have no problem walking up to a table of fourteen people and making conversation as if I had known them my whole life.
I give credit to “The Party” for putting me in a situation where I had no choice but to smile and carry on. I know now that not everyone will like me, and that’s okay. I’ve learned that, while I will always treat people with respect and dignity, their behavior toward me may be more about their own personal circumstances rather than anything I have done. Today, when I see “The Party” being seated in my section, I still flinch a little inside, but then I pull back my shoulders, lift my chin and march up to them ask, “What’s it going to be today, ladies.”
 """


In [49]:
prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n{essay}'
structured_model.invoke(prompt)

EvaluationSchema(score=9.0, feedback='This essay demonstrates excellent language quality. The narrative is compelling, drawing the reader in with vivid descriptions and a strong, relatable voice. The progression of the protagonist\'s personal growth is clearly articulated, moving from shyness and self-doubt to confidence and understanding. Vocabulary is appropriate and effective, and sentence structures are varied, contributing to a smooth and engaging reading experience. The use of dialogue effectively brings the characters to life. There are only minor stylistic points that could be refined, such as the phrasing in "march up to them ask, \'What\'s it going to be today, ladies.\'" which could be improved to "march up to them and ask." Overall, it\'s a very well-written and impactful piece.')

In [50]:
class UPSCState(TypedDict):
    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str
    individual_scores: Annotated[list[int] , operator.add]  # we use this beacuse at the time the all work parallely and we want to add the scores together to get the final score if not doing there is possiblity of overwrite    
    avg_score: float

In [51]:
def evaluate_Language(state:UPSCState) -> UPSCState:
    prompt = f'Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n{state["essay"]}'
    result = structured_model.invoke(prompt)      
    return {'individual_scores': [result.score] , 'language_feedback': result.feedback}

In [52]:
def evaluate_analysis(state: UPSCState):
    prompt = f'Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n{state["essay"]}'
    output = structured_model.invoke(prompt)
    return {'analysis_feedback': output.feedback, "individual_score": [output.score]}

In [53]:
def evaluate_thought(state: UPSCState):
    prompt = f'Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n{state["essay"]}'
    output = structured_model.invoke(prompt)
    return {'clarity_feedback': output.feedback, "individual_score": [output.score]}

In [54]:
def final_evaluation(state: UPSCState):
    total_score = sum(state['individual_scores'])
    avg_score = total_score / len(state['individual_scores'])
    overall_feedback = f"Overall Score: {avg_score:.2f}/10\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}"
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score} 

In [55]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate Language' , evaluate_Language)

graph.add_node('evaluate_analysis', evaluate_analysis)

graph.add_node('evaluate_thought', evaluate_thought)

graph.add_node('final_evaluation',final_evaluation)


graph.add_edge(START, 'evaluate Language')
graph.add_edge(START, 'evaluate_analysis')  
graph.add_edge(START, 'evaluate_thought')
graph.add_edge('evaluate Language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')
graph.add_edge('final_evaluation', END)

workflow = graph.compile()


In [59]:
intial_state = {
    'essay': essay,
}
workflow.invoke(intial_state)


{'essay': ' While bringing out the orders for my party of six, I set down a cup of soup in front of one of the women. Virginia glanced down at the clam chowder, and then scowled up at me.\n“Hannah, I asked for a half cup of soup,” she said, sounding as if the world was ending. “You are always so slow. I don’t know how you always mess up our orders.”\nI politely mumbled an apology, hoping the rest of her group did not notice how upset I was or how personally I took her comment. It was only a cup of soup, and an honest mistake, but I felt like such a disaster.\nResidents at the high-end senior living facility where I waited tables several nights a week expected nothing less than a five-star dining experience and my small mistake was not to be tolerated.\nWhen I first started working at Shannondell last summer, I was already shy and couldn’t stand the idea that someone didn’t like me. Early on, when residents would scold or criticize me, I felt like crawling under a rock. Even when I clea